# Train the virality scorer (DistilBERT detector → RoBERTa ranker)

Fine-tunes two models **sequentially** on the preprocessed YouTube trending
titles, forming a detect-then-rank cascade:

1. **DistilBERT detector** — binary classification on `viral_label`
   (top-quartile vs bottom-quartile in-category engagement). A fast filter for
   "is this viral-worthy at all?".
2. **RoBERTa ranker** — regression on `viral_score` (0–1 engagement percentile).
   Orders the survivors finely.

**Approach B:** we train *in-domain* on raw titles. At inference we won't feed
raw transcript lines (different domain) — we'll generate a candidate title from
each transcript segment and score *that*, so the scorer always sees title-like
text. So no title normalization here.

Inputs: `processed/{train,val,test}.parquet` from `preprocess_youtube_trending.ipynb`.

In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "torch", "datasets", "accelerate",
                "scikit-learn", "scipy", "pandas"], check=True)

CompletedProcess(args=['/Users/angie/miniforge3/envs/sam2/bin/python', '-m', 'pip', 'install', '-q', 'transformers', 'torch', 'datasets', 'accelerate', 'scikit-learn', 'scipy', 'pandas'], returncode=0)

## Config

In [2]:
import os

def _find(name):
    for p in [name, os.path.join("youtube_title_trending", name)]:
        if os.path.exists(p):
            return p
    return name

PROC_DIR = os.path.dirname(_find(os.path.join("processed", "train.parquet")))

DETECTOR_MODEL = "distilbert-base-uncased"
RANKER_MODEL   = "roberta-base"
DETECTOR_OUT   = "models/distilbert-detector"
RANKER_OUT     = "models/roberta-ranker"

MAX_LEN    = 64    # titles are short
EPOCHS     = 3
BATCH_SIZE = 16
LR         = 2e-5
LIMIT_TRAIN = None  # set an int for a quick smaller run
print("data dir:", PROC_DIR)

data dir: processed


In [3]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Device:", DEVICE)

train_df = pd.read_parquet(os.path.join(PROC_DIR, "train.parquet"))
val_df   = pd.read_parquet(os.path.join(PROC_DIR, "val.parquet"))
test_df  = pd.read_parquet(os.path.join(PROC_DIR, "test.parquet"))
if LIMIT_TRAIN:
    train_df = train_df.head(LIMIT_TRAIN)
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")

/Users/angie/miniforge3/envs/sam2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
train 7323 | val 915 | test 916


In [4]:
def make_dataset(df, text_col, label_col, tokenizer):
    """DataFrame -> tokenized HF Dataset with a `labels` column."""
    d = Dataset.from_pandas(
        df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "labels"}),
        preserve_index=False,
    )
    return d.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                 batched=True, remove_columns=["text"])

## Stage 1 — DistilBERT detector (binary `viral_label`)

Train only on the clear cases (label 0/1; the middle-band rows are NaN and dropped).

In [5]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.special import softmax

det_train = train_df.dropna(subset=["viral_label"]).copy()
det_val   = val_df.dropna(subset=["viral_label"]).copy()
for d in (det_train, det_val):
    d["viral_label"] = d["viral_label"].astype(int)
print(f"detector train {len(det_train)} | val {len(det_val)}")

det_tok = AutoTokenizer.from_pretrained(DETECTOR_MODEL)
det_ds_train = make_dataset(det_train, "video_title", "viral_label", det_tok)
det_ds_val   = make_dataset(det_val,   "video_title", "viral_label", det_tok)

def cls_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=1)[:, 1]
    preds = logits.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds),
            "f1": f1_score(labels, preds),
            "auc": roc_auc_score(labels, probs)}

det_model = AutoModelForSequenceClassification.from_pretrained(DETECTOR_MODEL, num_labels=2)
det_args = TrainingArguments(
    output_dir=DETECTOR_OUT, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR, weight_decay=0.01, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="auc", logging_steps=50, report_to="none")
det_trainer = Trainer(
    model=det_model, args=det_args, train_dataset=det_ds_train, eval_dataset=det_ds_val,
    data_collator=DataCollatorWithPadding(det_tok), processing_class=det_tok,
    compute_metrics=cls_metrics)
det_trainer.train()
det_trainer.save_model(DETECTOR_OUT)
det_tok.save_pretrained(DETECTOR_OUT)
print("detector val metrics:", det_trainer.evaluate())

detector train 3674 | val 472


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11748.09it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/Users/angie/miniforge3/envs/sam2/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true 

Epoch,Training Loss,Validation Loss,Accuracy,F1,Auc
1,0.507606,0.494162,0.756356,0.751620,0.846640
2,0.355087,0.494679,0.783898,0.785714,0.859613
3,0.314641,0.521603,0.794492,0.797495,0.862528


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.50it/s]
/Users/angie/miniforge3/envs/sam2/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 10.31it/s]
/Users/angie/miniforge3/envs/sam2/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 11.50it/s]
/Users/angie/miniforge3/envs/sam2/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1,Auc
0.314641,0.521603,3,0.794492,0.797495,0.862528


detector val metrics: {'eval_loss': 0.5216031074523926, 'eval_accuracy': 0.7944915254237288, 'eval_f1': 0.7974947807933194, 'eval_auc': 0.8625281151596942}


## Stage 2 — RoBERTa ranker (regress `viral_score`)

Trains on **all** rows (full 0–1 range) so it learns to order across the whole
spectrum, not just the extremes. Regression head: `num_labels=1`, MSE loss.

In [6]:
from scipy.stats import spearmanr

rank_train = train_df.copy(); rank_train["viral_score"] = rank_train["viral_score"].astype("float32")
rank_val   = val_df.copy();   rank_val["viral_score"]   = rank_val["viral_score"].astype("float32")

rank_tok = AutoTokenizer.from_pretrained(RANKER_MODEL)
rank_ds_train = make_dataset(rank_train, "video_title", "viral_score", rank_tok)
rank_ds_val   = make_dataset(rank_val,   "video_title", "viral_score", rank_tok)

def reg_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.squeeze()
    return {"spearman": spearmanr(preds, labels).correlation,
            "mse": float(np.mean((preds - labels) ** 2))}

rank_model = AutoModelForSequenceClassification.from_pretrained(RANKER_MODEL, num_labels=1)
rank_model.config.problem_type = "regression"
rank_args = TrainingArguments(
    output_dir=RANKER_OUT, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR, weight_decay=0.01, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="spearman", greater_is_better=True,
    logging_steps=50, report_to="none")
rank_trainer = Trainer(
    model=rank_model, args=rank_args, train_dataset=rank_ds_train, eval_dataset=rank_ds_val,
    data_collator=DataCollatorWithPadding(rank_tok), processing_class=rank_tok,
    compute_metrics=reg_metrics)
rank_trainer.train()
rank_trainer.save_model(RANKER_OUT)
rank_tok.save_pretrained(RANKER_OUT)
print("ranker val metrics:", rank_trainer.evaluate())

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2877.04it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/Users/angie/miniforge3/envs/sam2/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argume

Epoch,Training Loss,Validation Loss,Spearman,Mse
1,0.073445,0.068874,0.479330,0.068874
2,0.061033,0.064751,0.524050,0.064751
3,0.050937,0.062584,0.541795,0.062584


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.30it/s]
/Users/angie/miniforge3/envs/sam2/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]
/Users/angie/miniforge3/envs/sam2/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.69it/s]
/Users/angie/miniforge3/envs/sam2/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Spearman,Mse
0.050937,0.062584,3,0.541795,0.062584


ranker val metrics: {'eval_loss': 0.06258384883403778, 'eval_spearman': 0.5417950298252832, 'eval_mse': 0.06258384883403778}


## Evaluate the ranker on the held-out test set

Spearman + nDCG@10 of the predicted `viral_score` order vs. the true order.

In [7]:
from sklearn.metrics import ndcg_score

test_ds = make_dataset(test_df.assign(viral_score=test_df["viral_score"].astype("float32")),
                       "video_title", "viral_score", rank_tok)
pred = rank_trainer.predict(test_ds).predictions.squeeze()
true = test_df["viral_score"].to_numpy()
print("test Spearman:", round(spearmanr(pred, true).correlation, 4))
print("test nDCG@10:  ", round(ndcg_score([true], [pred], k=10), 4))

Map: 100%|██████████| 916/916 [00:00<00:00, 12859.20 examples/s]
/Users/angie/miniforge3/envs/sam2/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


test Spearman: 0.5692
test nDCG@10:   0.7823


## Cascade demo: detect → rank on example titles

In [8]:
examples = [
    "You won't BELIEVE what happened next 😱",
    "Q3 2024 quarterly earnings conference call",
    "I survived 100 days in hardcore Minecraft",
    "How to file your taxes: a step by step guide",
    "This $1 meal changed my life",
]

def score(titles):
    di = det_tok(titles, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(det_model.device)
    with torch.no_grad():
        det_p = torch.softmax(det_model(**di).logits, dim=1)[:, 1].tolist()
    ri = rank_tok(titles, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(rank_model.device)
    with torch.no_grad():
        rank_s = rank_model(**ri).logits.squeeze(-1).tolist()
    return pd.DataFrame({"title": titles, "detector_p_viral": det_p, "ranker_score": rank_s}) \
             .sort_values("ranker_score", ascending=False).round(4)

score(examples)

,title,detector_p_viral,ranker_score
0,You won't BELIEVE what happened next 😱,0.6816,0.5428
1,Q3 2024 quarterly earnings conference call,0.1544,0.4825
4,This $1 meal changed my life,0.3622,0.4791
3,How to file your taxes: a step by step guide,0.6368,0.4614
2,I survived 100 days in hardcore Minecraft,0.2089,0.3859
